# Stage 3 — Checking the filled-in locations

`pipeline/fill_locations.py` filled missing coordinates by matching addresses (street + zip first, then street only), using the most recent request's coordinates, and **skipping addresses whose known coordinates are spread more than 200 m apart**.

Output: `data/interim/located.parquet`. This notebook checks what was filled and how accurate it is. **Read-only.**

In [1]:
import sys
import duckdb
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "pipeline"))
from fill_locations import street_key_sql, MAX_SPREAD_M  # reuse the exact same key logic

S = (ROOT / "data/interim/standardized.parquet").as_posix()
L = (ROOT / "data/interim/located.parquet").as_posix()
con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
q = lambda sql: con.sql(sql).df()

## 1. Result overview

In [2]:
q(f'''SELECT coalesce(location_source, '(none)') AS location_source, count(*) AS n,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
     FROM '{L}' GROUP BY 1 ORDER BY n DESC''')

,location_source,n,pct
0,original,10701478,82.20
1,(none),2234683,17.16
2,address_match,83002,0.64


Of the **97,230** rows that had an address but no coordinates, about 78% were filled. The rest stay empty, along with the ~2.2M rows that had no address at all; Stage 4 moves them to their own table.

### Original coordinates were not changed

In [3]:
q(f'''SELECT count(*) AS original_rows,
            count(*) FILTER (WHERE s.latitude = l.latitude AND s.longitude = l.longitude) AS unchanged
     FROM '{S}' s JOIN '{L}' l USING (service_request_num)
     WHERE s.latitude IS NOT NULL''')

,original_rows,unchanged
0,10707011,10701478


### Filled rows by year

In [4]:
q(f'''SELECT year(created_date) AS yr,
            count(*) FILTER (WHERE location_source = 'address_match') AS filled,
            count(*) FILTER (WHERE location_source IS NULL AND address IS NOT NULL) AS address_but_not_filled
     FROM '{L}' GROUP BY 1 HAVING filled + address_but_not_filled > 0 ORDER BY 1''')

,yr,filled,address_but_not_filled
0,2010,58,14
1,2011,1,13
2,2012,4,7
3,2013,2,4
4,2014,34277,590
5,2015,11474,5724
6,2016,10252,5038
7,2017,10589,5041
8,2018,16343,3330
9,2019,1,0


## 2. What the cleaned matching key looks like

The key is built only for matching; the original `address` column is unchanged.

In [5]:
examples = pd.DataFrame({"address": [
    "3414  PARK-HEIGHTS AV , BALTIMORE, MD",
    "3414 PARK HEIGHTS AVENUE, Baltimore City, 21215",
    "1020 North Carrollton Avenue, 21217",
    "SPRINGDALE AVE & DENISON ST",
    "DENISON STREET & SPRINGDALE AVENUE, BC, 21216",
]})
con.register("ex", examples)
q(f"SELECT address, {street_key_sql('address')} AS street_key FROM ex")

,address,street_key
0,"3414 PARK-HEIGHTS AV , BALTIMORE, MD",3414 PARK HEIGHTS AVE
1,"3414 PARK HEIGHTS AVENUE, Baltimore City, 21215",3414 PARK HEIGHTS AVE
2,"1020 North Carrollton Avenue, 21217",1020 N CARROLLTON AVE
3,SPRINGDALE AVE & DENISON ST,DENISON ST & SPRINGDALE AVE
4,"DENISON STREET & SPRINGDALE AVENUE, BC, 21216",DENISON ST & SPRINGDALE AVE


Different spellings of the same place end up with the **same key**: different abbreviations, extra spaces, punctuation, town/zip suffixes, and the order of the two streets in an intersection.

## 3. Sample of filled rows

In [6]:
q(f'''SELECT service_request_num, year(created_date) AS yr, address, street_key,
            round(latitude, 5) AS lat, round(longitude, 5) AS lon
     FROM '{L}' WHERE location_source = 'address_match' ORDER BY random() LIMIT 10''')

,service_request_num,yr,address,street_key,lat,lon
0,15-00010040,2015,"2679 DULANY ST, 21223",2679 DULANY ST,39.27926,-76.65840
1,14-00956296,2014,"514 N PAYSON ST, 21223",514 N PAYSON ST,39.29412,-76.64914
2,14-00485521,2014,"439 E LAFAYETTE AVE, 21228",439 E LAFAYETTE AVE,39.30992,-76.60952
3,18-00086383,2018,"5720 CEDELLA AV , BALTIMORE, MD",5720 CEDELLA AVE,39.33276,-76.53732
4,16-00134257,2016,308 KINGSTON RD,308 KINGSTON RD,39.28543,-76.70692
5,14-00210450,2014,"2015 MCHENRY ST, 21223",2015 MCHENRY ST,39.28401,-76.64917
6,18-00636125,2018,"3911 DUDLEY AV , BALTIMORE, MD",3911 DUDLEY AVE,39.31733,-76.56116
7,14-00640099,2014,"1378 N CALHOUN ST, 21217",1378 N CALHOUN ST,39.30528,-76.64070
8,17-00131207,2017,5401 SAGRA RD,5401 SAGRA RD,39.35495,-76.59217
9,15-00161397,2015,"2 FARMHURST RD, PIKESVILLE, 21208",2 FARMHURST RD,39.36316,-76.72755


## 4. Rows that could NOT be filled — why?

In [7]:
q(f'''WITH known AS (SELECT DISTINCT street_key FROM '{L}' WHERE location_source = 'original')
     SELECT CASE WHEN k.street_key IS NULL THEN 'address never seen with coordinates'
                 ELSE 'address seen, but coordinates spread > {MAX_SPREAD_M} m' END AS reason,
            count(*) AS n, any_value(l.address) AS example
     FROM '{L}' l LEFT JOIN known k USING (street_key)
     WHERE l.location_source IS NULL AND l.address IS NOT NULL
     GROUP BY 1 ORDER BY n DESC''')

,reason,n,example
0,address never seen with coordinates,16397,MONROE ST
1,"address seen, but coordinates spread > 200 m",3364,FRANKFURST AVE


## 5. Accuracy test

**Idea:** take 50,000 random rows that *do* have real coordinates, hide them, fill them with the same method using only the *other* rows, and measure how far the guess is from the truth.

Simplification: this test uses street-only matching (no zip step), so it is a slightly pessimistic estimate.

In [8]:
con.sql(f'''
CREATE OR REPLACE TABLE known AS
SELECT service_request_num, created_date, latitude, longitude, street_key
FROM '{L}' WHERE location_source = 'original' AND street_key <> '';

CREATE OR REPLACE TABLE test AS SELECT * FROM known USING SAMPLE 50000 ROWS (reservoir, 42);

CREATE OR REPLACE TABLE lookup AS
SELECT street_key,
       arg_max(latitude, created_date) AS fill_lat, arg_max(longitude, created_date) AS fill_lon,
       max(111320 * sqrt(power(latitude - ml, 2) + power((longitude - mo) * cos(radians(39.3)), 2))) AS spread_m
FROM (SELECT *, median(latitude) OVER w AS ml, median(longitude) OVER w AS mo
      FROM known ANTI JOIN test USING (service_request_num)   -- exclude the hidden rows
      WINDOW w AS (PARTITION BY street_key))
GROUP BY street_key;
''')
res = q(f'''
SELECT t.service_request_num,
       111320 * sqrt(power(t.latitude - l.fill_lat, 2) + power((t.longitude - l.fill_lon) * cos(radians(39.3)), 2)) AS error_m
FROM test t JOIN lookup l USING (street_key) WHERE l.spread_m <= {MAX_SPREAD_M}''')
print(f"Hidden rows: 50,000 | could be filled: {len(res):,} ({len(res)/500:.0f}%)")
res.error_m.describe(percentiles=[.5, .9, .95, .99]).round(1).to_frame("error (meters)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hidden rows: 50,000 | could be filled: 47,528 (95%)


,error (meters)
count,47528.0
mean,20.4
std,438.2
min,0.0
50%,0.0
90%,25.7
95%,44.4
99%,99.7
max,25523.7


In [9]:
bins = [0, 10, 25, 50, 100, 200, 1e9]
labels = ["< 10 m", "10–25 m", "25–50 m", "50–100 m", "100–200 m", "> 200 m"]
pd.cut(res.error_m, bins=bins, labels=labels, right=False).value_counts(sort=False).to_frame("rows").assign(
    pct=lambda d: (100 * d.rows / d.rows.sum()).round(1))

,rows,pct
error_m,,
< 10 m,37693,79.3
10–25 m,4915,10.3
25–50 m,2966,6.2
50–100 m,1483,3.1
100–200 m,395,0.8
> 200 m,76,0.2


For scale: a Baltimore rowhouse block is roughly 100–150 m long, and neighborhood (NSA) areas are several hundred meters to a few km across. Errors of a few dozen meters will almost never put a request in the wrong neighborhood.